In [1]:
import kagglehub
path = kagglehub.dataset_download("vaibhavpawar123/flower-recognition")

100%|██████████| 225M/225M [00:01<00:00, 150MB/s]

Extracting files...


In [2]:
print(path)

/root/.cache/kagglehub/datasets/vaibhavpawar123/flower-recognition/versions/1


In [8]:
data_dir = "/root/.cache/kagglehub/datasets/vaibhavpawar123/flower-recognition/versions/1/flowers"

In [9]:
from PIL import Image
import os

clean_dir = "cleaned_dataset"

os.makedirs(clean_dir, exist_ok=True)

for label in os.listdir(data_dir):
    class_dir = os.path.join(data_dir, label)
    if not os.path.isdir(class_dir):
        continue

    clean_class_dir = os.path.join(clean_dir, label)
    os.makedirs(clean_class_dir, exist_ok=True)

    for file in os.listdir(class_dir):
        file_path = os.path.join(class_dir, file)
        try:
            img = Image.open(file_path).convert("RGB")  # convert any image to RGB
            img.save(os.path.join(clean_class_dir, file))
        except:
            print(f"Skipping corrupted image: {file_path}")


In [28]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from keras.layers import Dense , Conv2D , MaxPooling2D , Flatten , BatchNormalization
from PIL import Image


In [15]:
data_dir = "/content/cleaned_dataset"

**Cleaning Dirty Images**

In [16]:
from PIL import Image
import os

print("Cleaning invalid images...")

for label in os.listdir(data_dir):
    class_dir = os.path.join(data_dir, label)
    if not os.path.isdir(class_dir):
        continue

    for file in os.listdir(class_dir):
        file_path = os.path.join(class_dir, file)
        try:
            img = Image.open(file_path)
            img.verify()  # check for corruption

            # Re-open to check mode
            img = Image.open(file_path)
            if img.mode not in ["L", "RGB", "RGBA"]:
                print(f"Removing image with unsupported channels ({img.mode}): {file_path}")
                os.remove(file_path)

        except Exception as e:
            print(f"Removing corrupted image: {file_path}")
            os.remove(file_path)


Cleaning invalid images...


In [17]:
# import tensorflow as tf
# import numpy as np

batch_size = 32
img_size = (256, 256)
validation_split = 0.2
seed = 42

# Load training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",  # or "grayscale" if you want 2D arrays
    batch_size=batch_size,
    image_size=img_size,
    shuffle=True,
    seed=seed,
    validation_split=validation_split,
    subset="training"
)

# Load validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    labels="inferred",
    label_mode="int",
    color_mode="rgb",  # or "grayscale"
    batch_size=batch_size,
    image_size=img_size,
    shuffle=True,
    seed=seed,
    validation_split=validation_split,
    subset="validation"
)




Found 4317 files belonging to 5 classes.
Using 3454 files for training.
Found 4317 files belonging to 5 classes.
Using 863 files for validation.


In [18]:
train_ds.file_paths

['/content/cleaned_dataset/tulip/21055041762_5d0a066948_n.jpg',
 '/content/cleaned_dataset/rose/18464052396_6327841e65_n.jpg',
 '/content/cleaned_dataset/rose/15699509054_d3e125286f_n.jpg',
 '/content/cleaned_dataset/rose/8241471746_5d81fdd3c0_n.jpg',
 '/content/cleaned_dataset/tulip/19868509504_b2fc173274_n.jpg',
 '/content/cleaned_dataset/dandelion/8738317694_eca2ce3bfc_n.jpg',
 '/content/cleaned_dataset/tulip/4290566894_c7f061583d_m.jpg',
 '/content/cleaned_dataset/rose/3465443774_6b0c75a3b1_n.jpg',
 '/content/cleaned_dataset/rose/5335935851_53634d6aed_n.jpg',
 '/content/cleaned_dataset/dandelion/4336536446_e635f48f2e.jpg',
 '/content/cleaned_dataset/sunflower/4186808407_06688641e2_n.jpg',
 '/content/cleaned_dataset/dandelion/2489438981_4eb60ef98f_m.jpg',
 '/content/cleaned_dataset/rose/3621011057_0d03bd171b_n.jpg',
 '/content/cleaned_dataset/daisy/4669117051_ce61e91b76.jpg',
 '/content/cleaned_dataset/rose/921138131_9e1393eb2b_m.jpg',
 '/content/cleaned_dataset/sunflower/1356862194

In [19]:
def process(image, label):
    image = tf.cast(image / 255., tf.float32)  # Normalize image values to [0, 1] range
    return image, label

# Apply the process function to the training and validation datasets
train_ds = train_ds.map(process)
val_ds = val_ds.map(process)

In [38]:
model = Sequential()

model.add(Conv2D(32 , kernel_size=(3,3) , activation = 'relu' , padding='valid' , input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2) , strides=2 , padding = 'valid'))

model.add(Conv2D(32 , kernel_size=(3,3) , activation = 'relu' , padding='valid' , input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2) , strides=2 , padding = 'valid'))

model.add(Conv2D(32 , kernel_size=(3,3) , activation = 'relu' , padding='valid' , input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2) , strides=2 , padding = 'valid'))

model.add(Conv2D(32 , kernel_size=(3,3) , activation = 'relu' , padding='valid' , input_shape=(256,256,3)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2,2) , strides=2 , padding = 'valid'))

model.add(Flatten())

model.add(Dense(20 , activation='relu' , kernel_regularizer=tf.keras.regularizers.L2(0.05)))
model.add(BatchNormalization())

model.add(Dense(5 , activation='softmax'))

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_11 (Conv2D)              │ (None, 254, 254, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 254, 254, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 127, 127, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 125, 125, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 125, 125, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 60, 60, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 60, 60, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 30, 30, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 20)             │       125,460 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 20)             │            80 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 5)              │           105 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 154,797 (604.68 KB)

 Trainable params: 154,501 (603.52 KB)

 Non-trainable params: 296 (1.16 KB)

In [39]:
model.compile(loss='sparse_categorical_crossentropy' , metrics = ['accuracy'] , optimizer=keras.optimizers.Adam(learning_rate=0.01))

In [ ]:
history = model.fit(train_ds , validation_data=val_ds , epochs = 10 , batch_size=32)

Epoch 1/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 196s 2s/step - accuracy: 0.4236 - loss: 3.8221 - val_accuracy: 0.2433 - val_loss: 2.1353
Epoch 2/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 193s 2s/step - accuracy: 0.4981 - loss: 1.7559 - val_accuracy: 0.3812 - val_loss: 1.7686
Epoch 3/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 190s 2s/step - accuracy: 0.5274 - loss: 1.6339 - val_accuracy: 0.3476 - val_loss: 2.3693
Epoch 4/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.5377 - loss: 1.6681 - val_accuracy: 0.5805 - val_loss: 1.5370
Epoch 5/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 190s 2s/step - accuracy: 0.5516 - loss: 1.6415 - val_accuracy: 0.5133 - val_loss: 1.8542
Epoch 6/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.5570 - loss: 1.7754 - val_accuracy: 0.5087 - val_loss: 1.8333
Epoch 7/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.6032 - loss: 1.7564 - val_accuracy: 0.4716 - val_loss: 1.8898
Epoch 8/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.6001 - loss: 1.6377 - val_accu

In [ ]:
plt.plot(history.history['accuracy'] , color = 'red' , label = 'train')
plt.plot(history.history['val_accuracy'] , color = 'blue' , label = 'validation')

In [ ]:
plt.plot(history.history['loss'] , color = 'red' , label = 'train')
plt.plot(history.history['val_loss'] , color = 'blue' , label = 'validation')